# brier - M5.4b: L2-only re-run after the calibration fix

The first M5.4 run gave L2 accuracy 0.807 but ECE 0.461: the L2 temperature was fitted against
smoothed targets that capped confidence near 0.46 with 20 classes. The fix (commit `faa2cc9`)
uses Platt targets only for two classes and plain temperature scaling beyond. This notebook
re-runs **only L2** (raw / L0 / L1 are unchanged) on the same pinned model, data and split.

**Runtime:** A100, about 3-5 min including the model download.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRIER_COMMIT = "faa2cc9613f4787b96f772fb54e0d9e5c414ac91"  # L2 calibration fix
MODEL = "Qwen/Qwen3-1.7B"
REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"  # same pinned model as before
OUT = "/content/results_l2"

In [ ]:
!git clone -q https://github.com/mohdUwaish59/brier.git /content/brier
!git -C /content/brier checkout -q {BRIER_COMMIT}
!pip install -q -e "/content/brier[hf]"

In [ ]:
!python -m brier.bench run --model {MODEL} --revision {REVISION} --dtype bfloat16 \
    --batch-size 64 --levels L2 --l2-budgets 100,200,300 --out {OUT}

In [ ]:
import glob
import json

with open(glob.glob(f"{OUT}/*.json")[0]) as f:
    res = json.load(f)


def fmt(v):
    return f"{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]"


print("Before the fix (budget 300): accuracy 0.807, ECE 0.461, NLL 1.364")
print("L1 reference (same split):   accuracy 0.668, ECE 0.094, NLL 1.204")
print()
print("| labels | layer | solver | alpha | T | at bound | accuracy | ece | nll | brier |")
print("|---|---|---|---|---|---|---|---|---|---|")
for budget, e in res["l2_curve"].items():
    m = e["metrics"]
    print(
        f"| {budget} | {e['layer']} | {e['solver']} | {e['alpha']} | {e['temperature']:.4f} | "
        f"{e['temperature_at_bound']} | {fmt(m['accuracy'])} | {fmt(m['ece'])} | "
        f"{fmt(m['nll'])} | {fmt(m['brier'])} |"
    )
print()
print("flip rate (300):", fmt(res["levels"]["L2"]["flip_rate"]))
print("AURC (300):", fmt(res["levels"]["L2"]["aurc"]))
print("wall seconds:", res["timing"]["wall_seconds"])

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/brier_m54b_l2", "zip", "/content", "results_l2")
files.download("/content/brier_m54b_l2.zip")